# Teste de Extração: PIB VAB Agropecuária

Extrai o Valor Adicionado Bruto da Agropecuária via API SIDRA (Tabela 5938).

## URL da API SIDRA (março/2026)

| Tabela | Variável | Descrição | URL Padrão |
|--------|----------|-----------|------------|
| **5938** | 513 | VAB Agropecuária | `https://apisidra.ibge.gov.br/values/t/5938/n6/all/v/513/p/{ANO}` |

## Status dos Dados

- **Série disponível:** 2002 a 2023
- **Última liberação:** dados 2022-2023 (dez/2025)
- **2024+:** Ainda não disponível (PIB municipal demora ~2 anos para consolidação)
- **Nível territorial:** n6 = Municípios

## Fonte Oficial

- **IBGE - PIB dos Municípios:** https://www.ibge.gov.br/estatisticas/economicas/contas-nacionais/9088-produto-interno-bruto-dos-municipios.html

## Colunas da Resposta

| Código | Descrição |
|--------|-----------|
| D1C | Ano |
| D2C | Código do município |
| D2N | Nome do município |
| D3C | Setor (variável) |
| V | Valor (R$) |

## Troubleshooting

| Problema | Solução |
|----------|----------|
| JSON vazio | Ano sem dados disponíveis (ex.: 2024+) |
| Erro 503 | Retry com backoff exponencial |
| sidrapy instável | Usar requests direto |
| Header como dado | Remover linha com `iloc[1:]` |

In [1]:
import requests
import pandas as pd
import time
from pathlib import Path
from tqdm import tqdm
from typing import Optional

# ====================== CONFIG ======================
DATA_DIR = Path("data")
BRONZE_DIR = DATA_DIR / "bronze"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

class ExtratorPIBMunicipal:
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) PIB-Downloader/2026'
        })

    def baixar_ano(self, ano: int) -> Optional[pd.DataFrame]:
        """Baixa VAB Agropecuária de um ano específico com retry automático."""
        url = f"https://apisidra.ibge.gov.br/values/t/5938/n6/all/v/513/p/{ano}"
        for tentativa in range(5):
            try:
                r = self.session.get(url, timeout=45)
                r.raise_for_status()
                data = r.json()
                
                if not data or not isinstance(data, list) or len(data) <= 1:
                    print(f"Sem dados para {ano} (resposta vazia ou apenas header)")
                    return None
                
                df = pd.DataFrame(data)
                print(f"Sucesso {ano}: {len(df):,} municípios carregados")
                return df
                
            except Exception as e:
                print(f"Tentativa {tentativa+1}/5 - {ano}: {e}")
                time.sleep(2 ** tentativa)  # backoff exponencial: 1s → 2s → 4s → 8s → 16s
        
        print(f"❌ Falha total ao baixar {ano}")
        return None

    def extrair_e_salvar(self, anos: list[int]):
        """Extrai e salva dados particionados por ano."""
        for ano in tqdm(anos, desc="Baixando VAB Agropecuária Municipal"):
            df = self.baixar_ano(ano)
            if df is not None and not df.empty:
                # Limpa e prepara (remove header se veio)
                if 'D1C' in df.columns and df['D1C'].iloc[0] == 'Ano':
                    df = df.iloc[1:].copy()
                
                out_dir = BRONZE_DIR / "pib_vab_agro" / f"ano={ano}"
                out_dir.mkdir(parents=True, exist_ok=True)
                df.to_parquet(out_dir / "dados.parquet", index=False)
                print(f"Salvo: {out_dir / 'dados.parquet'}")
            
            time.sleep(1.5)  # respeita rate limit

# ====================== EXECUÇÃO ======================
if __name__ == "__main__":
    # Anos reais com dados (até 2023 no momento)
    anos_disponiveis = list(range(2010, 2024))  # 2024 não tem ainda → script pula automaticamente
    # anos = [2020, 2021, 2022, 2023]  # exemplo conservador

    extrator = ExtratorPIBMunicipal()
    extrator.extrair_e_salvar(anos_disponiveis)
    
    print("Finalizado! Dados salvos em data/bronze/pib_vab_agro/ano=XXXX/")

Baixando VAB Agropecuária Municipal:   0%|          | 0/14 [00:00<?, ?it/s]

Sucesso 2010: 5,571 municípios carregados
Salvo: data/bronze/pib_vab_agro/ano=2010/dados.parquet


Baixando VAB Agropecuária Municipal:   7%|▋         | 1/14 [00:08<01:48,  8.32s/it]

Sucesso 2011: 5,571 municípios carregados
Salvo: data/bronze/pib_vab_agro/ano=2011/dados.parquet


Baixando VAB Agropecuária Municipal:  14%|█▍        | 2/14 [00:16<01:40,  8.39s/it]

...

Sucesso 2022: 5,571 municípios carregados
Salvo: data/bronze/pib_vab_agro/ano=2022/dados.parquet


Baixando VAB Agropecuária Municipal:  93%|█████████▎| 13/14 [01:54<00:07,  7.38s/it]

Sucesso 2023: 5,571 municípios carregados
Salvo: data/bronze/pib_vab_agro/ano=2023/dados.parquet


Baixando VAB Agropecuária Municipal: 100%|██████████| 14/14 [01:57<00:00,  8.36s/it]

Finalizado! Dados salvos em data/bronze/pib_vab_agro/ano=XXXX/
